# Week 5 — Generate experimental materials

**Research task:** Generate two framings from fixed policy facts and inspect semantic differences that could become causal confounds.

**Python introduced:** f-strings, named call arguments, integers, floats, temperature, output limits, latency and token metadata.

Work through input → messages → route → call → raw return → parsed output → check. Predict each output before running its cell. The assessed routine below is the same routine printed in the coursebook and task file.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session05/session05_treatment_generation.ipynb)

Colab supports the OpenRouter route only. Local JupyterLab or VS Code is canonical because it can also reach Ollama.

In [ ]:
# Colab setup: clone the public repository when running in Colab.
import os as setup_os
import subprocess as setup_subprocess
from pathlib import Path as SetupPath
if SetupPath('/content').exists():
    setup_repo = SetupPath('/content/GenAI_Soc2026')
    if not setup_repo.exists():
        setup_subprocess.run(['git','clone','https://github.com/cjbarrie/GenAI_Soc2026.git',str(setup_repo)], check=True)
    setup_os.chdir(setup_repo / 'workbook' / 'session05')
print('Working folder:', SetupPath.cwd())

## Load the course settings and SDKs

**Input:** installed Python packages, `config/course_models.json`, and—if it is not already set—the hidden OpenRouter key. **Operations:** `import` makes an installed tool available; `Path.cwd()` gives Python the current folder; the `while` block walks upward until it finds the course configuration; `json.loads(...)` turns the file's JSON text into a dictionary; square brackets retrieve the two model names. **Output:** `HOSTED_MODEL` and `LOCAL_MODEL` are strings. `getpass(...)` accepts the key without echoing it. The folder-search code is supplied setup and is not assessed.


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

## Choose a route and store the fixed facts and changing frame

The facts and frame are separate strings so the exercise can change one while holding the other constant. `temperature` is a float and `maximum_output_tokens` is an integer. They are recorded as model-call inputs, not treated as substantive properties of the message.


In [ ]:
import time

ROUTE = "ollama"  # change to "openrouter" if preferred
shared_facts = (
    "The proposal would create a pathway to permanent legal status for undocumented "
    "immigrants who meet residency and background-check requirements."
)
frame = "rights"
temperature = 0.7
max_tokens = 140

## Construct the prompt with an f-string

The leading `f` allows values inside braces to be inserted into a string. `{frame}` and `{facts}` are replaced by their current values. The output is one complete prompt; printing it is how we check that the intended frame changed and the factual material did not.


In [ ]:
prompt = (
    f"Write one 70–90 word survey message using a {frame} frame. "
    f"Use only these facts and return only the message:\n\n{shared_facts}"
)
messages = [{"role": "user", "content": prompt}]
print(prompt)

## Make one timed call and preserve route-specific metadata

`time.perf_counter()` records a clock value before and after the call; subtraction gives elapsed seconds. The selected branch supplies model, messages, temperature and output limit. It then retrieves the candidate text and, where available, token-use metadata. Latency and token counts describe this run, not the candidate's validity.


In [ ]:
started = time.perf_counter()
if ROUTE == "openrouter":
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response = client.chat.send(
            model=HOSTED_MODEL, messages=messages,
            temperature=temperature, max_tokens=max_tokens,
        )
    raw_output = response.choices[0].message.content
    input_tokens = getattr(response.usage, "prompt_tokens", None)
    output_tokens = getattr(response.usage, "completion_tokens", None)
else:
    response = ollama.chat(
        model=LOCAL_MODEL, messages=messages,
        options={"temperature": temperature, "num_predict": max_tokens},
    )
    raw_output = response.message.content
    input_tokens = response.prompt_eval_count
    output_tokens = response.eval_count
latency_seconds = round(time.perf_counter() - started, 2)

print("Raw candidate:", raw_output.strip())
print("Input tokens:", input_tokens)
print("Output tokens:", output_tokens)
print("Latency seconds:", latency_seconds)

## Save the design settings beside the candidate

The dictionary stores the changing frame, fixed facts, route, model, settings, elapsed time and generated text together. This is the output needed for the later causal-design review. The named change asks for a second candidate; comparison should isolate intended framing from other semantic differences.


In [ ]:
candidate_record = {
    "route": ROUTE,
    "frame": frame,
    "shared_facts": shared_facts,
    "temperature": temperature,
    "max_tokens": max_tokens,
    "raw_output": raw_output.strip(),
}
print(candidate_record)

# ONE CHANGE: change frame from "rights" to "economics" and nothing else.

## Methodological check

Holding the prompt variables fixed does not guarantee that the generated messages differ only in framing. Compare facts, certainty, tone, length and implied beneficiaries before treating them as experimental stimuli.
## Completion recording

Run the rights and economics frames through one chosen route. Explain the f-string and every named call argument, then compare the candidates for semantic confounds and state what a human pretest must establish.

Explain every input and output aloud. Never show the shared key.